# Decision Trees from Scratch

**Interview-focused reference notebook** -- CART algorithm, Gini/entropy, pruning, feature importance.

## Core Intuition

A decision tree recursively partitions the feature space using axis-aligned splits. At each node, we choose the feature and threshold that best separates the classes (or reduces variance for regression).

**Key concepts:**
- **Impurity measures:** Gini index, entropy (information gain), variance (regression)
- **Splitting:** Greedy search over all features and thresholds
- **Stopping criteria:** max depth, min samples per leaf, min impurity decrease
- **Pruning:** Remove branches that don't help generalization

**Complexity:**
- Training: $O(n \cdot d \cdot \log n)$ per split (sorting), $O(n \cdot d \cdot n \cdot \log n)$ worst case for full tree
- Prediction: $O(\text{depth})$ per sample

**Impurity measures:**
- Gini: $G = 1 - \sum_k p_k^2$ (probability of misclassification)
- Entropy: $H = -\sum_k p_k \log_2 p_k$ (information content)
- Information gain: $IG = H(\text{parent}) - \sum_c \frac{n_c}{n} H(\text{child}_c)$
- Note: information gain = mutual information between feature split and label

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Impurity Measures

In [ ]:
def gini_impurity(y):
    """Gini index: 1 - sum(p_k^2). Range: [0, 1-1/K]."""
    if len(y) == 0:
        return 0
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return 1 - np.sum(probs ** 2)

def entropy(y):
    """Shannon entropy: -sum(p_k * log2(p_k)). Range: [0, log2(K)]."""
    if len(y) == 0:
        return 0
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    probs = probs[probs > 0]  # avoid log(0)
    return -np.sum(probs * np.log2(probs))

def variance_reduction(y):
    """Variance for regression trees."""
    if len(y) == 0:
        return 0
    return np.var(y)

# Visualize impurity measures for binary classification
p_range = np.linspace(0.001, 0.999, 200)

gini_vals = 1 - p_range**2 - (1-p_range)**2
entropy_vals = -p_range * np.log2(p_range) - (1-p_range) * np.log2(1-p_range)
misclass_vals = 1 - np.maximum(p_range, 1-p_range)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p_range, gini_vals, 'b-', linewidth=2, label='Gini impurity')
ax.plot(p_range, entropy_vals, 'r-', linewidth=2, label='Entropy (scaled)')
ax.plot(p_range, misclass_vals, 'g--', linewidth=2, label='Misclassification error')
ax.set_xlabel('P(class=1)')
ax.set_ylabel('Impurity')
ax.set_title('Impurity Measures for Binary Classification')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axvline(x=0.5, color='k', linestyle=':', alpha=0.3)
plt.tight_layout()
plt.show()

print("Gini and entropy are very similar in practice.")
print("Gini is slightly faster to compute (no log).")
print("Misclassification error is piecewise linear -- not smooth enough for greedy optimization.")

## 2. Synthetic Data

In [ ]:
from sklearn.datasets import make_classification, make_moons

# Iris-like synthetic data (3 classes)
from sklearn.datasets import make_blobs
X_iris, y_iris = make_blobs(n_samples=300, centers=3, n_features=2,
                             random_state=42, cluster_std=1.0)

# Nonlinear data (moons)
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=42)

# Regression data
X_reg = np.sort(np.random.uniform(0, 10, 200)).reshape(-1, 1)
y_reg = np.sin(X_reg.ravel()) + 0.2 * np.random.randn(200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
scatter = axes[0].scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap='viridis', s=30, alpha=0.7, edgecolors='k')
axes[0].set_title('3-Class (Iris-like)')
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='RdBu', s=30, alpha=0.7, edgecolors='k')
axes[1].set_title('Moons (Nonlinear)')
axes[2].scatter(X_reg, y_reg, s=20, alpha=0.6)
axes[2].set_title('Regression')
plt.tight_layout()
plt.show()

## 3. CART Implementation

CART (Classification and Regression Trees) uses binary splits:
- Classification: minimize Gini or entropy after split
- Regression: minimize variance after split

In [ ]:
class TreeNode:
    """A node in the decision tree."""
    def __init__(self, feature=None, threshold=None, left=None, right=None,
                 value=None, impurity=None, n_samples=None):
        self.feature = feature        # feature index for splitting
        self.threshold = threshold    # threshold value for splitting
        self.left = left              # left child (feature <= threshold)
        self.right = right            # right child (feature > threshold)
        self.value = value            # prediction value (leaf node)
        self.impurity = impurity      # impurity at this node
        self.n_samples = n_samples    # number of samples at this node
    
    def is_leaf(self):
        return self.value is not None


class DecisionTreeClassifier:
    """CART decision tree classifier."""
    
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1,
                 criterion='gini', min_impurity_decrease=0.0):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.min_impurity_decrease = min_impurity_decrease
        self.root = None
    
    def _impurity(self, y):
        if self.criterion == 'gini':
            return gini_impurity(y)
        elif self.criterion == 'entropy':
            return entropy(y)
    
    def _information_gain(self, y, y_left, y_right):
        """Compute information gain from a split."""
        n = len(y)
        n_left = len(y_left)
        n_right = len(y_right)
        if n_left == 0 or n_right == 0:
            return 0
        parent_impurity = self._impurity(y)
        child_impurity = (n_left / n) * self._impurity(y_left) + (n_right / n) * self._impurity(y_right)
        return parent_impurity - child_impurity
    
    def _best_split(self, X, y):
        """Find the best feature and threshold to split on."""
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        n, d = X.shape
        
        for feature in range(d):
            # Get unique sorted values for this feature
            thresholds = np.unique(X[:, feature])
            # Use midpoints between consecutive values
            if len(thresholds) > 1:
                thresholds = (thresholds[:-1] + thresholds[1:]) / 2
            
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                
                if left_mask.sum() < self.min_samples_leaf or right_mask.sum() < self.min_samples_leaf:
                    continue
                
                gain = self._information_gain(y, y[left_mask], y[right_mask])
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold, best_gain
    
    def _build_tree(self, X, y, depth=0):
        """Recursively build the tree."""
        n_samples = len(y)
        n_classes = len(np.unique(y))
        current_impurity = self._impurity(y)
        
        # Stopping conditions (pre-pruning)
        if (n_classes == 1 or
            n_samples < self.min_samples_split or
            (self.max_depth is not None and depth >= self.max_depth)):
            # Leaf node: majority class
            classes, counts = np.unique(y, return_counts=True)
            value = classes[np.argmax(counts)]
            return TreeNode(value=value, impurity=current_impurity, n_samples=n_samples)
        
        # Find best split
        best_feature, best_threshold, best_gain = self._best_split(X, y)
        
        if best_feature is None or best_gain < self.min_impurity_decrease:
            classes, counts = np.unique(y, return_counts=True)
            value = classes[np.argmax(counts)]
            return TreeNode(value=value, impurity=current_impurity, n_samples=n_samples)
        
        # Split
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = ~left_mask
        
        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return TreeNode(feature=best_feature, threshold=best_threshold,
                        left=left_child, right=right_child,
                        impurity=current_impurity, n_samples=n_samples)
    
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.n_features_ = X.shape[1]
        self.root = self._build_tree(X, y)
        return self
    
    def _predict_single(self, x, node):
        """Traverse the tree to make a prediction."""
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_single(x, node.left)
        else:
            return self._predict_single(x, node.right)
    
    def predict(self, X):
        return np.array([self._predict_single(x, self.root) for x in X])

# Test
tree = DecisionTreeClassifier(max_depth=5, criterion='gini').fit(X_iris, y_iris)
acc = np.mean(tree.predict(X_iris) == y_iris)
print(f"Decision Tree accuracy (Iris-like, depth=5): {acc:.4f}")

tree_moons = DecisionTreeClassifier(max_depth=8, criterion='gini').fit(X_moons, y_moons)
acc_moons = np.mean(tree_moons.predict(X_moons) == y_moons)
print(f"Decision Tree accuracy (Moons, depth=8): {acc_moons:.4f}")

## 4. Decision Tree for Regression

In [ ]:
class DecisionTreeRegressor:
    """CART decision tree for regression."""
    
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None
    
    def _variance_reduction(self, y, y_left, y_right):
        n = len(y)
        if len(y_left) == 0 or len(y_right) == 0:
            return 0
        return np.var(y) - (len(y_left)/n * np.var(y_left) + len(y_right)/n * np.var(y_right))
    
    def _best_split(self, X, y):
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])
            if len(thresholds) > 1:
                thresholds = (thresholds[:-1] + thresholds[1:]) / 2
            
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                
                if left_mask.sum() < self.min_samples_leaf or right_mask.sum() < self.min_samples_leaf:
                    continue
                
                gain = self._variance_reduction(y, y[left_mask], y[right_mask])
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold, best_gain
    
    def _build_tree(self, X, y, depth=0):
        n_samples = len(y)
        
        if (n_samples < self.min_samples_split or
            (self.max_depth is not None and depth >= self.max_depth) or
            np.var(y) < 1e-10):
            return TreeNode(value=np.mean(y), n_samples=n_samples)
        
        best_feature, best_threshold, best_gain = self._best_split(X, y)
        
        if best_feature is None or best_gain <= 0:
            return TreeNode(value=np.mean(y), n_samples=n_samples)
        
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = ~left_mask
        
        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return TreeNode(feature=best_feature, threshold=best_threshold,
                        left=left_child, right=right_child, n_samples=n_samples)
    
    def fit(self, X, y):
        self.root = self._build_tree(X, y)
        return self
    
    def _predict_single(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_single(x, node.left)
        else:
            return self._predict_single(x, node.right)
    
    def predict(self, X):
        return np.array([self._predict_single(x, self.root) for x in X])

# Test regression tree
reg_tree = DecisionTreeRegressor(max_depth=5).fit(X_reg, y_reg)
y_pred = reg_tree.predict(X_reg)
mse = np.mean((y_pred - y_reg) ** 2)
print(f"Regression tree MSE (depth=5): {mse:.4f}")

## 5. Visualize Decision Boundaries and Regression Fits

In [ ]:
def plot_tree_boundary(X, y, depths, title_prefix=''):
    fig, axes = plt.subplots(1, len(depths), figsize=(5*len(depths), 4))
    if len(depths) == 1:
        axes = [axes]
    
    for ax, d in zip(axes, depths):
        model = DecisionTreeClassifier(max_depth=d).fit(X, y)
        h = 0.05
        x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
        y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
        Z = model.predict(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
        ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=20, alpha=0.7, edgecolors='k', linewidths=0.5)
        acc = np.mean(model.predict(X) == y)
        ax.set_title(f'{title_prefix}Depth={d} (acc={acc:.3f})')
    return fig

plot_tree_boundary(X_iris, y_iris, [1, 3, 5, 10], 'Iris-like: ')
plt.tight_layout()
plt.show()

plot_tree_boundary(X_moons, y_moons, [1, 3, 5, 15], 'Moons: ')
plt.tight_layout()
plt.show()

print("Decision boundaries are always axis-aligned (a key limitation).")
print("Deep trees overfit -- they create tiny regions to capture noise.")

In [ ]:
# Regression tree visualization
X_plot_reg = np.linspace(0, 10, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, d in zip(axes, [1, 3, 5, 15]):
    model = DecisionTreeRegressor(max_depth=d).fit(X_reg, y_reg)
    y_pred = model.predict(X_plot_reg)
    
    ax.scatter(X_reg, y_reg, s=15, alpha=0.5, c='blue')
    ax.plot(X_plot_reg, y_pred, 'r-', linewidth=2)
    ax.plot(X_plot_reg, np.sin(X_plot_reg), 'g--', alpha=0.5, label='True')
    mse = np.mean((model.predict(X_reg) - y_reg)**2)
    ax.set_title(f'Depth={d} (MSE={mse:.3f})')
    ax.legend(fontsize=8)

plt.suptitle('Regression Tree: Piecewise Constant Approximation', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Tree Structure Visualization

In [ ]:
def print_tree(node, depth=0, prefix='Root'):
    """Print the tree structure."""
    indent = '  ' * depth
    if node.is_leaf():
        print(f"{indent}{prefix}: Leaf -> class={node.value} (n={node.n_samples})")
    else:
        print(f"{indent}{prefix}: feature[{node.feature}] <= {node.threshold:.3f} "
              f"(n={node.n_samples}, impurity={node.impurity:.3f})")
        print_tree(node.left, depth + 1, 'L')
        print_tree(node.right, depth + 1, 'R')

small_tree = DecisionTreeClassifier(max_depth=3, criterion='gini').fit(X_iris, y_iris)
print("Tree Structure (depth=3):")
print_tree(small_tree.root)

## 7. Feature Importance

Impurity-based feature importance: sum of impurity decreases at each split, weighted by the number of samples reaching that node.

In [ ]:
def compute_feature_importance(tree, n_features):
    """Compute impurity-based feature importance."""
    importances = np.zeros(n_features)
    
    def _accumulate(node, total_samples):
        if node.is_leaf():
            return
        # Weighted impurity decrease
        left_impurity = node.left.impurity if node.left.impurity is not None else 0
        right_impurity = node.right.impurity if node.right.impurity is not None else 0
        n_left = node.left.n_samples
        n_right = node.right.n_samples
        n = node.n_samples
        
        weighted_decrease = (n / total_samples) * (
            node.impurity - (n_left/n * left_impurity + n_right/n * right_impurity)
        )
        importances[node.feature] += weighted_decrease
        
        _accumulate(node.left, total_samples)
        _accumulate(node.right, total_samples)
    
    _accumulate(tree.root, tree.root.n_samples)
    # Normalize
    if importances.sum() > 0:
        importances /= importances.sum()
    return importances

# Multi-feature dataset
from sklearn.datasets import make_classification
X_fi, y_fi = make_classification(n_samples=500, n_features=8, n_informative=3,
                                  n_redundant=2, random_state=42)

tree_fi = DecisionTreeClassifier(max_depth=8).fit(X_fi, y_fi)
importances = compute_feature_importance(tree_fi, X_fi.shape[1])

fig, ax = plt.subplots(figsize=(8, 5))
idx = np.argsort(importances)[::-1]
ax.bar(range(len(importances)), importances[idx], color='steelblue')
ax.set_xticks(range(len(importances)))
ax.set_xticklabels([f'Feature {i}' for i in idx], rotation=45)
ax.set_ylabel('Feature Importance')
ax.set_title('Impurity-Based Feature Importance')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 8. Post-Pruning (Cost-Complexity Pruning)

Cost-complexity pruning minimizes: $R_\alpha(T) = R(T) + \alpha |T|$

where $R(T)$ is the misclassification rate, $|T|$ is the number of leaves, and $\alpha$ is the complexity parameter. Larger $\alpha$ = simpler tree.

In [ ]:
def count_leaves(node):
    """Count the number of leaf nodes."""
    if node.is_leaf():
        return 1
    return count_leaves(node.left) + count_leaves(node.right)

def tree_error(node, X, y):
    """Compute misclassification rate."""
    if node.is_leaf():
        return np.sum(y != node.value)
    left_mask = X[:, node.feature] <= node.threshold
    errors = 0
    if left_mask.sum() > 0:
        errors += tree_error(node.left, X[left_mask], y[left_mask])
    if (~left_mask).sum() > 0:
        errors += tree_error(node.right, X[~left_mask], y[~left_mask])
    return errors

def prune_tree(node, X, y, alpha):
    """Cost-complexity pruning."""
    if node.is_leaf():
        return node
    
    # Recursively prune children first
    left_mask = X[:, node.feature] <= node.threshold
    right_mask = ~left_mask
    
    if left_mask.sum() > 0:
        node.left = prune_tree(node.left, X[left_mask], y[left_mask], alpha)
    if right_mask.sum() > 0:
        node.right = prune_tree(node.right, X[right_mask], y[right_mask], alpha)
    
    # Check if we should prune this node
    # Cost with children: R(subtree) + alpha * |leaves|
    subtree_error = tree_error(node, X, y)
    subtree_leaves = count_leaves(node)
    subtree_cost = subtree_error / len(y) + alpha * subtree_leaves
    
    # Cost as leaf: misclassification + alpha * 1
    classes, counts = np.unique(y, return_counts=True)
    majority_class = classes[np.argmax(counts)]
    leaf_error = np.sum(y != majority_class) / len(y)
    leaf_cost = leaf_error + alpha
    
    if leaf_cost <= subtree_cost:
        # Prune: replace subtree with leaf
        return TreeNode(value=majority_class, impurity=None, n_samples=len(y))
    
    return node

# Show effect of pruning
import copy

big_tree = DecisionTreeClassifier(max_depth=20).fit(X_moons, y_moons)
print(f"Before pruning: {count_leaves(big_tree.root)} leaves, "
      f"accuracy = {np.mean(big_tree.predict(X_moons) == y_moons):.4f}")

for alpha in [0.0, 0.01, 0.02, 0.05, 0.1]:
    pruned_root = prune_tree(copy.deepcopy(big_tree.root), X_moons, y_moons, alpha)
    temp_tree = DecisionTreeClassifier()
    temp_tree.root = pruned_root
    acc = np.mean(temp_tree.predict(X_moons) == y_moons)
    n_leaves = count_leaves(pruned_root)
    print(f"alpha={alpha:.2f}: {n_leaves:>3} leaves, accuracy = {acc:.4f}")

## 9. Gini vs Entropy -- Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, criterion in zip(axes, ['gini', 'entropy']):
    model = DecisionTreeClassifier(max_depth=5, criterion=criterion).fit(X_moons, y_moons)
    h = 0.05
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='RdBu', s=20, alpha=0.7, edgecolors='k', linewidths=0.5)
    acc = np.mean(model.predict(X_moons) == y_moons)
    ax.set_title(f'{criterion.capitalize()} (acc={acc:.3f}, leaves={count_leaves(model.root)})')

plt.tight_layout()
plt.show()
print("In practice, Gini and entropy give very similar trees.")
print("Gini tends to isolate the most frequent class in its own branch.")
print("Entropy tends to produce more balanced trees.")

## 10. Sklearn Comparison

In [ ]:
from sklearn.tree import DecisionTreeClassifier as SkTree
from sklearn.tree import DecisionTreeRegressor as SkTreeReg
from sklearn.metrics import accuracy_score, mean_squared_error

# Classification comparison
print("Classification Comparison:")
print(f"{'Dataset':<12} {'Depth':>6} {'Ours':>10} {'sklearn':>10}")
print("-" * 40)

for name, X, y in [('Iris-like', X_iris, y_iris), ('Moons', X_moons, y_moons)]:
    for d in [3, 5, 10]:
        our_tree = DecisionTreeClassifier(max_depth=d).fit(X, y)
        sk_tree = SkTree(max_depth=d, random_state=42).fit(X, y)
        our_acc = accuracy_score(y, our_tree.predict(X))
        sk_acc = sk_tree.score(X, y)
        print(f"{name:<12} {d:>6} {our_acc:>10.4f} {sk_acc:>10.4f}")

# Regression comparison
print("\nRegression Comparison (MSE):")
for d in [3, 5, 10]:
    our_reg = DecisionTreeRegressor(max_depth=d).fit(X_reg, y_reg)
    sk_reg = SkTreeReg(max_depth=d, random_state=42).fit(X_reg, y_reg)
    our_mse = mean_squared_error(y_reg, our_reg.predict(X_reg))
    sk_mse = mean_squared_error(y_reg, sk_reg.predict(X_reg))
    print(f"  Depth={d}: Ours={our_mse:.4f}, sklearn={sk_mse:.4f}")

## Interview Quick Reference

| Question | Answer |
|----------|--------|
| Gini vs entropy -- when does it matter? | Rarely matters in practice. Gini is slightly faster (no log). Entropy produces slightly more balanced trees. Both are concave. |
| How to handle continuous features? | Sort by feature value, try all midpoint thresholds. CART does this automatically. |
| How to handle missing values? | Surrogate splits (CART): find another feature that gives a similar split. Or: treat missing as a separate category. XGBoost learns a default direction. |
| Why are trees high-variance? | Small changes in data can lead to completely different splits at the root, cascading through the tree. This is why ensembles (RF, boosting) are preferred. |
| Information gain = mutual information? | Yes. $IG(Y; X_j) = H(Y) - H(Y|X_j)$ = how much knowing $X_j$ reduces uncertainty about $Y$. |
| Advantages of trees? | Interpretable, handles mixed types, no feature scaling needed, captures nonlinear interactions. |
| Disadvantages? | High variance, axis-aligned splits only, greedy (not globally optimal), prone to overfitting. |
| Pre-pruning vs post-pruning? | Pre: stop early (max_depth, min_samples). Post: grow full tree, then prune back. Post-pruning often gives better results but is more expensive. |